## langchain 기초

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : d:\gangsa\hanwha-agent


In [2]:
from langchain_anthropic import ChatAnthropic

# Claude Chat Model 생성 
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=500, 
)

response = llm.invoke(
    "RAG가 무엇인지 한 문장으로 설명해주세요."
)

print(response)
print(type(response))

content='# RAG (Retrieval-Augmented Generation)\n\n**외부 데이터베이스에서 관련 정보를 검색하여 가져온 후, 이를 바탕으로 AI 모델이 더 정확하고 최신의 답변을 생성하는 기술입니다.**' additional_kwargs={} response_metadata={'id': 'msg_011CfHvjfEfegxA9hmrbvqYx', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'end_turn', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 31, 'output_tokens': 90, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic'} id='lc_run--01a0c797-48bb-7db0-8d5d-609c3b2a875d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 31, 'output_tokens': 90, 'total_tokens': 121, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1

In [ ]:
llm.invoke(
    "RAG 이/가 무엇인지 비전공자에게 설명해주세요."
)
# RAG 무엇인지 설명해주세요. 
# Embedding 무엇인지 설명해주세요. 
# Vector DB 무엇인지 설명해주세요. 
# -> 공통된 부분을 {변수} 로 치환하여 템플릿처럼 prompt를 관리할 수 있다. 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

#1. Prompt를 만든다 
prompt = ChatPromptTemplate.from_template(
	"{topic}에 대해 비전공자도 이해할 수 있도록 쉽게 설명해주세요."
)

#2. LLM을 만든다 
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=500, 
)
#3. Prompt와 LLM 을 연결한다 
chain = prompt | llm 

#4. chain을 실행한다. 
response = chain.invoke({
    "topic": "RAG"
})

#5. Claude의 답변을 출력한다. 
print(response.content)


# RAG(검색 증강 생성)를 쉽게 설명해드릴게요

## 가장 간단한 비유

마치 **시험을 볼 때 교과서를 참고하며 푸는 것**처럼 생각하면 됩니다.

- ❌ AI가 머릿속으로만 답변 (환각, 오류 발생)
- ✅ RAG: AI가 자료를 찾아본 후 답변 (정확성 향상)

---

## 구체적인 예시

**일반 AI:**
```
Q: "2024년 삼성 실적은?"
A: (학습 데이터에만 의존, 최신 정보 없음)
```

**RAG 적용 AI:**
```
Q: "2024년 삼성 실적은?"
1️⃣ 최신 뉴스/자료 검색
2️⃣ 관련 정보 수집
3️⃣ 정보를 기반으로 답변
A: "2024년 삼성은..." (정확한 최신 정보)
```

---

## RAG의 3단계 흐름

| 단계 | 설명 |
|------|------|
| **검색(R)** | 질문과 관련된 자료를 찾는다 |
| **생성(G)** | 찾은 자료를 바탕으로 답변을 만든다 |
| **증강(A)** | 이 두 가지를 합쳐서 더 좋은 답변 |

---

## 실생활 예시

- 📚 **도서관 직원**: 책을 찾아주고(검색) → 그 내용으


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#1. Prompt를 만든다 
prompt = ChatPromptTemplate.from_template(
	"{topic}에 대해 비전공자도 이해할 수 있도록 쉽게 설명해주세요."
)

#2. LLM을 만든다 
llm = ChatAnthropic(
    model="claude-haiku-4-5",
    max_tokens=500, 
)

#3.AIMessage에서 문자열 추출을 위한 parser 
parser = StrOutputParser()

#4. Prompt와 LLM 을 연결한다 
chain = prompt | llm | parser 

#5. chain을 실행한다. 
response = chain.invoke({
    "topic": "RAG"
})

#6. Claude의 답변을 출력한다. 
print(response)
print(type(response))

# RAG(검색 증강 생성)을 쉽게 설명하면

## 🎯 핵심 개념
**AI가 답변할 때, 인터넷에서 정보를 먼저 찾아본 후 답변하는 방식**

---

## 📚 비유로 이해하기

### ❌ RAG 없이 (기존 방식)
```
당신: "2024년 최신 뉴스가 뭔가요?"
AI: "음... 제 학습 데이터는 2023년까지만 있어서..."
```
→ AI가 학습한 것만 알고, 최신 정보는 모름

### ✅ RAG 있이 (개선된 방식)
```
당신: "2024년 최신 뉴스가 뭔가요?"
AI: (먼저 인터넷 검색) → "오늘의 뉴스는... 입니다"
```
→ 최신 정보를 찾아서 답변

---

## 🔄 RAG의 작동 원리

```
1️⃣ 질문받기
   ↓
2️⃣ 관련 정보 검색 (외부 데이터베이스/인터넷)
   ↓
3️⃣ 찾은 정보 + 질문으로 답변 생성
   ↓
4️⃣ 사용자에게 답변 제공
```

---

## 💡 실제 예시

| 상황 | RAG 없음 | RAG 있음 |
|------|---------|---------|
| 회사의 내부 규정 질문 | "잘 몰라요" | 회사 문서 
<class 'langchain_core.messages.base.TextAccessor'>
